#### 발전소 구분 WS : 월성 KR : 고리 YK : 한빛 UJ : 한울
###### 월단위 원자력발전소 상태 : NuclearPlantStates
###### 분단위 실시간 폐수 수질현황: WasteWater
###### 10분단위 실시간 주변 방사선 량: RadioRate
###### 10분단위 실시간 온배수 현황: ThermalWasteWater
#### 본부 구분(2100:고리본부,2200:월성본부,2300:한빛본부,2400:한울본부,2800:새울본부)
###### 2024년 10월 이후 월단위 방사성 폐기물 발생량: RadioActiveWaste

In [1]:
%useLatestDescriptors
%use datetime
%use dataframe
%use ktor-client

In [2]:
val confFilePath = "/Users/unchil/Library/Application Support/Google/AndroidStudio2025.3.3/scratches/http-client.private.env.json"
val confDf = DataRow.readJson(path=confFilePath)

In [17]:

import kotlinx.datetime.format.byUnicodePattern
import kotlinx.datetime.format.FormatStringsInDatetimeFormats
import kotlinx.datetime.format.byUnicodePattern
import kotlin.time.Clock

fun loadKHNP_Service(url:String): DataFrame<*> {
    val now = Clock.System.now()
    val genNames = listOf("WS", "KR", "YK", "SU", "UJ")
    val rows = mutableListOf<DataFrame<*>>()
    val myCollectionTime = now.toLocalDateTime(TimeZone.of("Asia/Seoul")).format(LocalDateTime.Format { byUnicodePattern("yyyy-MM-dd HH:mm") })

    genNames.forEach { genName ->
        val urlPath = url + "&genName=${genName}"
        try {
            val df_json = DataFrame.readJson( http.get(urlPath).deserializeJson().jsonString.byteInputStream())
            val instanceDf =
                df_json.get("response").get("body").get("items").get("item")[0] as DataFrame<*>

            val updatedDf = instanceDf.add {
                "collectionTime" from { myCollectionTime }
                "genName" from { genName }
            }
            rows.add(updatedDf)
        }catch(e:Exception){
            println(e.localizedMessage)
            println(urlPath)
        }
    }

    return rows.concat()
}

##### 분단위 실시간 폐수 수질현황: WasteWater

In [ ]:
val url = "${confDf.KHNP.host}/${confDf.KHNP.suburl[1]}?serviceKey=${confDf.KHNP.key}"
val df = loadKHNP_Service(url)
df

In [8]:
val updatedDf = df.update { name }.with {
    val currentName = it.toString() // 현재 행의 name 값
    when {
        currentName.contains("FLW00") || currentName.contains("TM001") -> "TM001"
        currentName.contains("PHY00") || currentName.contains("TM002") -> "TM002"
        else -> it // 조건에 해당하지 않으면 원래 값 유지
    }
}
updatedDf

expl,name,time,value,collectionTime,genName
폐수수질-방류량(TM001),TM001,2026-04-22 13:37,105.699997,2026-04-22 13:37,WS
폐수수질-PH(TM002),TM002,2026-04-22 13:37,7.100000,2026-04-22 13:37,WS
폐수수질-방류량,TM001,2026-04-22 13:36,111.000000,2026-04-22 13:37,KR
폐수수질-PH,TM002,2026-04-22 13:36,7.300000,2026-04-22 13:37,KR
폐수수질-방류량(TM001),TM001,2026-03-30 06:58,0,2026-04-22 13:37,YK
폐수수질-PH(TM002),TM002,2026-03-30 06:58,0,2026-04-22 13:37,YK


In [9]:
val pivotedDf = updatedDf.pivot { name  }.groupBy { collectionTime and genName }.values { value and time }.flatten()
pivotedDf

collectionTime,genName,value,time,value1,time1
2026-04-22 13:37,WS,105.699997,2026-04-22 13:37,7.100000,2026-04-22 13:37
2026-04-22 13:37,KR,111.000000,2026-04-22 13:36,7.300000,2026-04-22 13:36
2026-04-22 13:37,YK,0,2026-03-30 06:58,0,2026-03-30 06:58


In [10]:
val renameDf = pivotedDf.rename(
    "value" to "tm001",
    "time" to "tm001_time",
    "value1" to "tm002",
    "time1" to "tm002_time"
)
renameDf

collectionTime,genName,tm001,tm001_time,tm002,tm002_time
2026-04-22 13:37,WS,105.699997,2026-04-22 13:37,7.100000,2026-04-22 13:37
2026-04-22 13:37,KR,111.000000,2026-04-22 13:36,7.300000,2026-04-22 13:36
2026-04-22 13:37,YK,0,2026-03-30 06:58,0,2026-03-30 06:58


###### 10분단위 실시간 온배수 현황: ThermalWasteWater

In [ ]:
val url3 = "${confDf.KHNP.host}/${confDf.KHNP.suburl[3]}?serviceKey=${confDf.KHNP.key}"
val df3 = loadKHNP_Service(url3)
df3

RM001 : 취수구-수온 , RM002 : 취수구-염분, RM005 : 배수구-수온, RM006 : 배수구-염분

In [12]:
val updatedDf3 = df3.update { name }.with {
    val currentName = it.toString() // 현재 행의 name 값
    when {
        currentName.contains("RM001")-> "RM001"
        currentName.contains("RM002")-> "RM002"
        currentName.contains("RM005")-> "RM005"
        currentName.contains("RM006")-> "RM006"
        else -> it // 조건에 해당하지 않으면 원래 값 유지
    }
}
updatedDf3

expl,name,time,value,collectionTime,genName
취수구-수온,RM001,2026-04-22 13:30,14.300000,2026-04-22 13:39,WS
취수구-염분,RM002,2026-04-22 13:30,34.200001,2026-04-22 13:39,WS
배수구-수온,RM005,2026-04-22 13:30,14.200000,2026-04-22 13:39,WS
배수구-염분,RM006,2026-04-22 13:30,32.900002,2026-04-22 13:39,WS
취수구-수온,RM001,2026-02-04 18:37,0,2026-04-22 13:39,KR
취수구-염분,RM002,2026-02-04 18:37,0,2026-04-22 13:39,KR
배수구-수온,RM005,2026-02-04 18:37,0,2026-04-22 13:39,KR
배수구-염분,RM006,2026-02-04 18:37,0,2026-04-22 13:39,KR
취수구-수온,RM001,2026-04-22 13:30,16.299999,2026-04-22 13:39,YK
취수구-염분,RM002,2026-04-22 13:30,32.400002,2026-04-22 13:39,YK


In [13]:
val pivotedDf3 = updatedDf3.pivot { name  }.groupBy { collectionTime and genName }.values { value and time }.flatten()
pivotedDf3

collectionTime,genName,value,time,value1,time1,value2,time2,value3,time3
2026-04-22 13:39,WS,14.300000,2026-04-22 13:30,34.200001,2026-04-22 13:30,14.200000,2026-04-22 13:30,32.900002,2026-04-22 13:30
2026-04-22 13:39,KR,0,2026-02-04 18:37,0,2026-02-04 18:37,0,2026-02-04 18:37,0,2026-02-04 18:37
2026-04-22 13:39,YK,16.299999,2026-04-22 13:30,32.400002,2026-04-22 13:30,24.000000,2026-04-22 13:30,32.299999,2026-04-22 13:30
2026-04-22 13:39,UJ,10.200000,2026-04-22 13:39,33.599998,2026-04-22 13:39,15.300000,2026-04-22 13:39,33.500000,2026-04-22 13:39


In [14]:
val renameDf3 = pivotedDf3.rename(
    "value" to "rm001",
    "time" to "rm001_time",
    "value1" to "rm002",
    "time1" to "rm002_time",
    "value2" to "rm005",
    "time2" to "rm005_time",
    "value3" to "rm006",
    "time3" to "rm006_time",
)
renameDf3

collectionTime,genName,rm001,rm001_time,rm002,rm002_time,rm005,rm005_time,rm006,rm006_time
2026-04-22 13:39,WS,14.300000,2026-04-22 13:30,34.200001,2026-04-22 13:30,14.200000,2026-04-22 13:30,32.900002,2026-04-22 13:30
2026-04-22 13:39,KR,0,2026-02-04 18:37,0,2026-02-04 18:37,0,2026-02-04 18:37,0,2026-02-04 18:37
2026-04-22 13:39,YK,16.299999,2026-04-22 13:30,32.400002,2026-04-22 13:30,24.000000,2026-04-22 13:30,32.299999,2026-04-22 13:30
2026-04-22 13:39,UJ,10.200000,2026-04-22 13:39,33.599998,2026-04-22 13:39,15.300000,2026-04-22 13:39,33.500000,2026-04-22 13:39


In [15]:
val convertedDf = renameDf3.convert {
    collectionTime and rm001_time and rm002_time and rm005_time and rm006_time
}.with {
    LocalDateTime.parse(it.toString().replace(" ", "T")).toInstant(TimeZone.UTC)
}

convertedDf

collectionTime,genName,rm001,rm001_time,rm002,rm002_time,rm005,rm005_time,rm006,rm006_time
2026-04-22T13:39:00Z,WS,14.300000,2026-04-22T13:30:00Z,34.200001,2026-04-22T13:30:00Z,14.200000,2026-04-22T13:30:00Z,32.900002,2026-04-22T13:30:00Z
2026-04-22T13:39:00Z,KR,0,2026-02-04T18:37:00Z,0,2026-02-04T18:37:00Z,0,2026-02-04T18:37:00Z,0,2026-02-04T18:37:00Z
2026-04-22T13:39:00Z,YK,16.299999,2026-04-22T13:30:00Z,32.400002,2026-04-22T13:30:00Z,24.000000,2026-04-22T13:30:00Z,32.299999,2026-04-22T13:30:00Z
2026-04-22T13:39:00Z,UJ,10.200000,2026-04-22T13:39:00Z,33.599998,2026-04-22T13:39:00Z,15.300000,2026-04-22T13:39:00Z,33.500000,2026-04-22T13:39:00Z


In [22]:
val checkTime = 20.minutes

val finalDf = convertedDf.filter {
    (collectionTime - rm001_time).absoluteValue <= checkTime &&
    (collectionTime - rm002_time).absoluteValue <= checkTime &&
    (collectionTime - rm005_time).absoluteValue <= checkTime &&
    (collectionTime - rm006_time).absoluteValue <= checkTime
}
finalDf

collectionTime,genName,rm001,rm001_time,rm002,rm002_time,rm005,rm005_time,rm006,rm006_time
2026-04-22T13:39:00Z,WS,14.300000,2026-04-22T13:30:00Z,34.200001,2026-04-22T13:30:00Z,14.200000,2026-04-22T13:30:00Z,32.900002,2026-04-22T13:30:00Z
2026-04-22T13:39:00Z,YK,16.299999,2026-04-22T13:30:00Z,32.400002,2026-04-22T13:30:00Z,24.000000,2026-04-22T13:30:00Z,32.299999,2026-04-22T13:30:00Z
2026-04-22T13:39:00Z,UJ,10.200000,2026-04-22T13:39:00Z,33.599998,2026-04-22T13:39:00Z,15.300000,2026-04-22T13:39:00Z,33.500000,2026-04-22T13:39:00Z
